In [42]:
# Cell 1: Imports
import gzip, os, struct, time, json, math, glob, random, platform, importlib.util
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.models import resnet18, ResNet18_Weights, resnet34, ResNet34_Weights, resnet50, ResNet50_Weights
from google.colab import drive
from tqdm.auto import tqdm

In [43]:
# Cell 2: Configuration (session B: separate output folder)
_R18 = dict(arch='r18', pretrained=True, size=112, epochs=15, batch=128, lr=0.01, wd=5e-4, ls=0.0, dropout=0.0,
            shift=2, rot=0.0, scale=0.0, bright=0.0, contrast=0.0)
_SMALL = dict(arch='small', pretrained=False, size=28, epochs=40, batch=128, lr=0.1, wd=5e-4, ls=0.0, dropout=0.2,
              shift=2, rot=0.0, scale=0.0, bright=0.0, contrast=0.0)
CFG = dict(
    data_dir='/content/drive/MyDrive/data',
    out_dir='/content/drive/MyDrive/models_b',
    sub_dir='/content/drive/MyDrive/models_b/unused',
    num_classes=11,
    train_counts=[1112, 767, 1920, 1493, 1082, 595, 1077, 604, 710, 688, 3327],
    valid_counts=[183, 264, 274, 202, 131, 90, 156, 100, 248, 237, 469],
    imagenet_mean=(0.485, 0.456, 0.406),
    imagenet_std=(0.229, 0.224, 0.225),
    momentum=0.9,
    amp=True,
    seeds=(0, 1, 2),
    la_tau=1.0,
    adopt_min_gain=0.01,
    bootstrap=2000,
    models=dict(
        r18_in=_R18,
        scratch=_SMALL,
        r18_in_lr003={**_R18, 'lr': 0.003},
        r18_in_lr03={**_R18, 'lr': 0.03},
        scratch_up56={**_SMALL, 'size': 56},
        r18_in_lr06={**_R18, 'lr': 0.06},
        r18_in_lr10={**_R18, 'lr': 0.1},
        r18_in_lr03_ep30={**_R18, 'lr': 0.03, 'epochs': 30},
        r34_in_lr03={**_R18, 'arch': 'r34', 'lr': 0.03},
        r18_in_lr03_ls={**_R18, 'lr': 0.03, 'ls': 0.1},
        r18_in_lr03_wd1e4={**_R18, 'lr': 0.03, 'wd': 1e-4},
        r18_in_lr03_ls_wd1e4={**_R18, 'lr': 0.03, 'ls': 0.1, 'wd': 1e-4},
        r18_in_lr03_mixup={**_R18, 'lr': 0.03, 'mix': 'mixup', 'mix_alpha': 0.2},
        r18_in_lr03_cutmix={**_R18, 'lr': 0.03, 'mix': 'cutmix', 'mix_alpha': 1.0},
        r50_in_lr03={**_R18, 'arch': 'r50', 'lr': 0.03},
    ),
    queue=['r18_in', 'scratch', 'r18_in_lr003', 'r18_in_lr03', 'scratch_up56',
           'r18_in_lr06', 'r18_in_lr10', 'r18_in_lr03_ep30', 'r34_in_lr03',
           'r18_in_lr03_ls', 'r18_in_lr03_wd1e4', 'r18_in_lr03_ls_wd1e4',
           'r18_in_lr03_mixup', 'r18_in_lr03_cutmix', 'r50_in_lr03'],
    comparisons=[('r18_in_lr03', 'r18_in_lr03_mixup', False, False),
                 ('r18_in_lr03', 'r18_in_lr03_cutmix', False, False),
                 ('r18_in_lr03', 'r50_in_lr03', False, False)],
    final_model=None,
    final_la=False,
    final_fit='train+valid',
    final_seeds=(0, 1, 2),
)

In [44]:
# Cell 3: Drive mount, environment record and seeding
drive.mount('/content/drive')
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
assert torch.cuda.is_available(), 'GPU runtime required: Runtime -> Change runtime type -> GPU'
dev = torch.device('cuda')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

os.makedirs(CFG['out_dir'], exist_ok=True)
os.makedirs(CFG['sub_dir'], exist_ok=True)
ENV = dict(python=platform.python_version(), torch=torch.__version__, torchvision=torchvision.__version__,
           numpy=np.__version__, cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
           gpu=torch.cuda.get_device_name(0),
           gpu_mem_gb=round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
           cudnn_tf32=torch.backends.cudnn.allow_tf32, matmul_tf32=torch.backends.cuda.matmul.allow_tf32,
           deterministic=True, amp_bf16=CFG['amp'])
with open(os.path.join(CFG['out_dir'], 'env.json'), 'w') as f:
    json.dump(ENV, f, indent=1)
print(json.dumps(ENV, indent=1))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{
 "python": "3.13.16",
 "torch": "2.11.0+cu130",
 "torchvision": "0.26.0+cu130",
 "numpy": "2.4.6",
 "cuda": "13.0",
 "cudnn": 92700,
 "gpu": "NVIDIA A100-SXM4-80GB",
 "gpu_mem_gb": 79.3,
 "cudnn_tf32": true,
 "matmul_tf32": false,
 "deterministic": true,
 "amp_bf16": true
}


In [45]:
# Cell 4: Data loading and integrity checks
def load_idx(path):
    with gzip.open(path, 'rb') as f:
        _, _, _, ndim = struct.unpack('>BBBB', f.read(4))
        shape = struct.unpack(f'>{ndim}I', f.read(4 * ndim))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(shape)

D = CFG['data_dir']
Xtr_np = load_idx(f'{D}/train-images-idx3-ubyte.gz')
Ytr_np = load_idx(f'{D}/train-labels-idx1-ubyte.gz').astype(np.int64)
Xva_np = load_idx(f'{D}/valid-images-idx3-ubyte.gz')
Yva_np = load_idx(f'{D}/valid-labels-idx1-ubyte.gz').astype(np.int64)
assert Xtr_np.shape == (13375, 28, 28) and Xva_np.shape == (2354, 28, 28)
assert np.bincount(Ytr_np, minlength=11).tolist() == CFG['train_counts']
assert np.bincount(Yva_np, minlength=11).tolist() == CFG['valid_counts']

def to_t(a):
    return torch.from_numpy(a.copy()).float().div(255).unsqueeze(1).to(dev)

Xtr, Xva = to_t(Xtr_np), to_t(Xva_np)
Ytr, Yva = torch.from_numpy(Ytr_np).to(dev), torch.from_numpy(Yva_np).to(dev)
PRIOR_TR = np.bincount(Ytr_np, minlength=11) / len(Ytr_np)
print('train', tuple(Xtr.shape), 'valid', tuple(Xva.shape), f'train mean {Xtr.mean().item():.4f} std {Xtr.std().item():.4f}')

train (13375, 1, 28, 28) valid (2354, 1, 28, 28) train mean 0.4953 std 0.2826


In [46]:
# Cell 5: Model definitions
DEPTHS = {'r18': (2, 2, 2, 2), 'r34': (3, 4, 6, 3)}
PRETRAINED = {'r18': (resnet18, ResNet18_Weights.IMAGENET1K_V1),
              'r34': (resnet34, ResNet34_Weights.IMAGENET1K_V1),
              'r50': (resnet50, ResNet50_Weights.IMAGENET1K_V1)}


class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.downsample = None
        if stride != 1 or cin != cout:
            self.downsample = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        return F.relu(y + (x if self.downsample is None else self.downsample(x)))


class Bottleneck(nn.Module):
    def __init__(self, cin, mid, stride):
        super().__init__()
        cout = mid * 4
        self.conv1 = nn.Conv2d(cin, mid, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid)
        self.conv2 = nn.Conv2d(mid, mid, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(mid)
        self.conv3 = nn.Conv2d(mid, cout, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(cout)
        self.downsample = None
        if stride != 1 or cin != cout:
            self.downsample = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = F.relu(self.bn2(self.conv2(y)))
        y = self.bn3(self.conv3(y))
        return F.relu(y + (x if self.downsample is None else self.downsample(x)))


class ResNetBody(nn.Module):
    out_dim = 512

    def __init__(self, depths):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        stages = [(64, 64, 1), (64, 128, 2), (128, 256, 2), (256, 512, 2)]
        for i, ((a, b, s), n) in enumerate(zip(stages, depths), 1):
            setattr(self, f'layer{i}', nn.Sequential(Block(a, b, s), *[Block(b, b, 1) for _ in range(n - 1)]))

    def forward(self, x):
        x = self.maxpool(F.relu(self.bn1(self.conv1(x))))
        return self.layer4(self.layer3(self.layer2(self.layer1(x)))).mean((2, 3))


class ResNet50Body(nn.Module):
    out_dim = 2048

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        cin = 64
        for i, (mid, n, s) in enumerate([(64, 3, 1), (128, 4, 2), (256, 6, 2), (512, 3, 2)], 1):
            setattr(self, f'layer{i}', nn.Sequential(Bottleneck(cin, mid, s), *[Bottleneck(mid * 4, mid, 1) for _ in range(n - 1)]))
            cin = mid * 4

    def forward(self, x):
        x = self.maxpool(F.relu(self.bn1(self.conv1(x))))
        return self.layer4(self.layer3(self.layer2(self.layer1(x)))).mean((2, 3))


class SmallResNet(nn.Module):
    out_dim = 128

    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.blocks = nn.Sequential(Block(32, 32, 1), Block(32, 64, 2), Block(64, 128, 2))

    def forward(self, x):
        return self.blocks(self.stem(x)).mean((2, 3))


class Net(nn.Module):
    def __init__(self, arch, num_classes, size, dropout, mean=None, std=None):
        super().__init__()
        self.size = size
        if arch == 'r50':
            self.body = ResNet50Body()
        elif arch in DEPTHS:
            self.body = ResNetBody(DEPTHS[arch])
        else:
            self.body = SmallResNet()
        ch = 1 if arch == 'small' else 3
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(self.body.out_dim, num_classes)
        self.register_buffer('mean', torch.tensor(mean if mean is not None else [0.0] * ch, dtype=torch.float32).view(1, ch, 1, 1))
        self.register_buffer('std', torch.tensor(std if std is not None else [1.0] * ch, dtype=torch.float32).view(1, ch, 1, 1))
        self.register_buffer('logit_bias', torch.zeros(num_classes))

    def forward(self, x):
        if x.shape[-1] != self.size:
            x = F.interpolate(x, size=(self.size, self.size), mode='bilinear', align_corners=False)
        x = (x.expand(-1, self.mean.shape[1], -1, -1) - self.mean) / self.std
        return self.fc(self.drop(self.body(x))) + self.logit_bias


def build_net(c, mean, std):
    net = Net(c['arch'], CFG['num_classes'], c['size'], c['dropout'], mean, std)
    if c['pretrained']:
        fn, w = PRETRAINED[c['arch']]
        sd = {k: v for k, v in fn(weights=w).state_dict().items() if not k.startswith('fc.')}
        net.body.load_state_dict(sd, strict=True)
    return net


for _name, _c in CFG['models'].items():
    _n = build_net({**_c, 'pretrained': False}, None, None)
    print(f"{_name:22s} params {sum(p.numel() for p in _n.parameters()):,}")
for _arch in sorted({c['arch'] for c in CFG['models'].values() if c['pretrained']}):
    _c = next(c for c in CFG['models'].values() if c['arch'] == _arch and c['pretrained'])
    _n = build_net(_c, CFG['imagenet_mean'], CFG['imagenet_std'])
    print(f'{PRETRAINED[_arch][1]} loaded into {_arch} body (strict)')
del _n

r18_in                 params 11,182,155
scratch                params 308,203
r18_in_lr003           params 11,182,155
r18_in_lr03            params 11,182,155
scratch_up56           params 308,203
r18_in_lr06            params 11,182,155
r18_in_lr10            params 11,182,155
r18_in_lr03_ep30       params 11,182,155
r34_in_lr03            params 21,290,315
r18_in_lr03_ls         params 11,182,155
r18_in_lr03_wd1e4      params 11,182,155
r18_in_lr03_ls_wd1e4   params 11,182,155
r18_in_lr03_mixup      params 11,182,155
r18_in_lr03_cutmix     params 11,182,155
r50_in_lr03            params 23,530,571
ResNet18_Weights.IMAGENET1K_V1 loaded into r18 body (strict)
ResNet34_Weights.IMAGENET1K_V1 loaded into r34 body (strict)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 184MB/s]


ResNet50_Weights.IMAGENET1K_V1 loaded into r50 body (strict)


In [47]:
# Cell 6: Augmentation, mixing, training and evaluation functions
def augment(x, c, g):
    n = x.shape[0]

    def u():
        return torch.rand(n, device=x.device, generator=g) * 2 - 1

    ang = u() * math.radians(c['rot'])
    sc = 1 + u() * c['scale']
    tx, ty = u() * c['shift'] * 2 / 28, u() * c['shift'] * 2 / 28
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    x = F.grid_sample(x, F.affine_grid(theta, list(x.shape), align_corners=False),
                      mode='bilinear', padding_mode='border', align_corners=False)
    if c['bright'] or c['contrast']:
        m = x.mean((1, 2, 3), keepdim=True)
        x = ((x - m) * (1 + u().view(-1, 1, 1, 1) * c['contrast']) + m + u().view(-1, 1, 1, 1) * c['bright']).clamp(0, 1)
    return x


def mix_batch(x, y, c, g, rng):
    lam = float(rng.beta(c['mix_alpha'], c['mix_alpha']))
    perm = torch.randperm(x.shape[0], device=x.device, generator=g)
    if c['mix'] == 'mixup':
        return lam * x + (1 - lam) * x[perm], y, y[perm], lam
    h, w = x.shape[-2:]
    rh, rw = int(round(h * math.sqrt(1 - lam))), int(round(w * math.sqrt(1 - lam)))
    cy, cx = int(rng.integers(0, h)), int(rng.integers(0, w))
    y1, y2 = max(cy - rh // 2, 0), min(cy + rh - rh // 2, h)
    x1, x2 = max(cx - rw // 2, 0), min(cx + rw - rw // 2, w)
    x = x.clone()
    x[:, :, y1:y2, x1:x2] = x[perm][:, :, y1:y2, x1:x2]
    lam = 1 - (y2 - y1) * (x2 - x1) / (h * w)
    return x, y, y[perm], lam


def param_groups(net, wd):
    dec = [p for p in net.parameters() if p.ndim > 1]
    nod = [p for p in net.parameters() if p.ndim <= 1]
    return [dict(params=dec, weight_decay=wd), dict(params=nod, weight_decay=0.0)]


def train_model(name, seed, X, Y):
    c = CFG['models'][name]
    mix = c.get('mix', 'none')
    set_seed(seed)
    if c['pretrained']:
        mean, std = CFG['imagenet_mean'], CFG['imagenet_std']
    else:
        mean, std = (X.mean().item(),), (X.std().item(),)
    net = build_net(c, mean, std).to(dev)
    opt = torch.optim.SGD(param_groups(net, c['wd']), lr=c['lr'], momentum=CFG['momentum'], nesterov=True)
    n, bs = len(X), c['batch']
    total = c['epochs'] * math.ceil(n / bs)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * min(s, total) / total)))
    g = torch.Generator(device=dev)
    g.manual_seed(seed)
    rng = np.random.default_rng(seed)
    t0 = time.time()
    bar = tqdm(range(c['epochs']), desc=f'{name} s{seed}', leave=False)
    for ep in bar:
        net.train()
        perm = torch.randperm(n, device=dev, generator=g)
        tl = torch.zeros((), device=dev)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            xb = augment(X[idx], c, g)
            yb = Y[idx]
            with torch.autocast('cuda', dtype=torch.bfloat16, enabled=CFG['amp']):
                if mix == 'none':
                    loss = F.cross_entropy(net(xb), yb, label_smoothing=c['ls'])
                else:
                    xm, ya, yb2, lam = mix_batch(xb, yb, c, g, rng)
                    out = net(xm)
                    loss = lam * F.cross_entropy(out, ya, label_smoothing=c['ls']) + \
                        (1 - lam) * F.cross_entropy(out, yb2, label_smoothing=c['ls'])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            tl += loss.detach().float() * len(idx)
        bar.set_postfix(loss=f'{tl.item() / n:.4f}', lr=f'{sched.get_last_lr()[0]:.2e}')
    bar.close()
    sec = time.time() - t0
    return net, dict(train_loss=tl.item() / n, sec=sec, sec_per_epoch=sec / c['epochs'],
                     params=sum(p.numel() for p in net.parameters()))


@torch.no_grad()
def get_logits(net, X, bs=1024):
    net.eval()
    return torch.cat([net(X[i:i + bs]).float() for i in range(0, len(X), bs)]).cpu().numpy()


def mf1(y, p, k=11):
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k)
    tp = np.diag(cm)
    return float((2 * tp / np.maximum(cm.sum(0) + cm.sum(1), 1)).mean())


def scores(y, p, k=11):
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k)
    tp = np.diag(cm)
    f1 = 2 * tp / np.maximum(cm.sum(0) + cm.sum(1), 1)
    return dict(macro_f1=float(f1.mean()), acc=float(tp.sum() / len(y)), f1=f1.round(4).tolist())


LA = -CFG['la_tau'] * np.log(PRIOR_TR)
print('functions ready; logit adjustment', LA.round(3).tolist())

functions ready; logit adjustment [2.487, 2.859, 1.941, 2.193, 2.515, 3.113, 2.519, 3.098, 2.936, 2.967, 1.391]


In [48]:
# Cell 7: Experiment runner (train split only, scored on valid at the final epoch)
RES_PATH = os.path.join(CFG['out_dir'], 'results.json')
RESULTS = json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}
for name in CFG['queue']:
    for seed in CFG['seeds']:
        rid = f'{name}_s{seed}'
        if rid in RESULTS:
            print('SKIP', rid)
            continue
        net, info = train_model(name, seed, Xtr, Ytr)
        lg = get_logits(net, Xva)
        np.save(os.path.join(CFG['out_dir'], f'logits_{rid}.npy'), lg)
        RESULTS[rid] = dict(model=name, seed=seed, cfg=CFG['models'][name], env_gpu=ENV['gpu'], **info,
                            raw=scores(Yva_np, lg.argmax(1)), la=scores(Yva_np, (lg + LA).argmax(1)))
        with open(RES_PATH, 'w') as f:
            json.dump(RESULTS, f, indent=1)
        r = RESULTS[rid]
        print(f"DONE {rid} | {info['sec']:.0f}s ({info['sec_per_epoch']:.1f}s/ep) | params {info['params']:,} | "
              f"macroF1 {r['raw']['macro_f1']:.4f} acc {r['raw']['acc']:.4f} | LA macroF1 {r['la']['macro_f1']:.4f}")
        del net
        torch.cuda.empty_cache()

SKIP r18_in_s0
SKIP r18_in_s1
SKIP r18_in_s2
SKIP scratch_s0
SKIP scratch_s1
SKIP scratch_s2
SKIP r18_in_lr003_s0
SKIP r18_in_lr003_s1
SKIP r18_in_lr003_s2
SKIP r18_in_lr03_s0
SKIP r18_in_lr03_s1
SKIP r18_in_lr03_s2
SKIP scratch_up56_s0
SKIP scratch_up56_s1
SKIP scratch_up56_s2
SKIP r18_in_lr06_s0
SKIP r18_in_lr06_s1
SKIP r18_in_lr06_s2
SKIP r18_in_lr10_s0
SKIP r18_in_lr10_s1
SKIP r18_in_lr10_s2
SKIP r18_in_lr03_ep30_s0
SKIP r18_in_lr03_ep30_s1
SKIP r18_in_lr03_ep30_s2
SKIP r34_in_lr03_s0
SKIP r34_in_lr03_s1
SKIP r34_in_lr03_s2
SKIP r18_in_lr03_ls_s0
SKIP r18_in_lr03_ls_s1
SKIP r18_in_lr03_ls_s2
SKIP r18_in_lr03_wd1e4_s0
SKIP r18_in_lr03_wd1e4_s1
SKIP r18_in_lr03_wd1e4_s2
SKIP r18_in_lr03_ls_wd1e4_s0
SKIP r18_in_lr03_ls_wd1e4_s1
SKIP r18_in_lr03_ls_wd1e4_s2


r18_in_lr03_mixup s0:   0%|          | 0/15 [00:00<?, ?it/s]

DONE r18_in_lr03_mixup_s0 | 24s (1.6s/ep) | params 11,182,155 | macroF1 0.8575 acc 0.9129 | LA macroF1 0.8573


r18_in_lr03_mixup s1:   0%|          | 0/15 [00:00<?, ?it/s]

DONE r18_in_lr03_mixup_s1 | 24s (1.6s/ep) | params 11,182,155 | macroF1 0.8760 acc 0.9214 | LA macroF1 0.8741


r18_in_lr03_mixup s2:   0%|          | 0/15 [00:00<?, ?it/s]

... (9 outputs left out)

r50_in_lr03 s1:   0%|          | 0/15 [00:00<?, ?it/s]

DONE r50_in_lr03_s1 | 56s (3.7s/ep) | params 23,530,571 | macroF1 0.8736 acc 0.9248 | LA macroF1 0.8757


r50_in_lr03 s2:   0%|          | 0/15 [00:00<?, ?it/s]

DONE r50_in_lr03_s2 | 55s (3.7s/ep) | params 23,530,571 | macroF1 0.8781 acc 0.9214 | LA macroF1 0.8790


In [49]:
# Cell 8: Results table, top confusions and paired comparisons (per-seed and ensemble; macro-F1, then accuracy, then size)
def runs_of(name):
    return sorted([r for r in RESULTS.values() if r['model'] == name], key=lambda r: r['seed'])


def preds_of(name, la):
    rs = runs_of(name)
    lgs = [np.load(os.path.join(CFG['out_dir'], f"logits_{name}_s{r['seed']}.npy")) for r in rs]
    return [r['seed'] for r in rs], [(l + (LA if la else 0)).argmax(1) for l in lgs], lgs


def top_confusions(y, p, k=11, n=6):
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k)
    np.fill_diagonal(cm, 0)
    idx = np.argsort(-cm, axis=None)[:n]
    return [f'{i // k}->{i % k}:{cm.flat[i]}' for i in idx]


def acc_of(y, p):
    return float(np.mean(y == p))


for name in [m for m in CFG['models'] if runs_of(m)]:
    rs = runs_of(name)
    seeds, _, lgs = preds_of(name, False)
    mf = [r['raw']['macro_f1'] for r in rs]
    ac = [r['raw']['acc'] for r in rs]
    mean_lg = np.mean(lgs, 0)
    print(f"{name:16s} seeds {seeds} | macroF1 {np.mean(mf):.4f}±{np.std(mf):.4f} | acc {np.mean(ac):.4f}±{np.std(ac):.4f} | "
          f"LA macroF1 {np.mean([r['la']['macro_f1'] for r in rs]):.4f} | ens macroF1 {mf1(Yva_np, mean_lg.argmax(1)):.4f} "
          f"acc {acc_of(Yva_np, mean_lg.argmax(1)):.4f} | params {rs[0]['params']:,} | {np.mean([r['sec_per_epoch'] for r in rs]):.1f}s/ep")
    print('   per-class F1', np.mean([r['raw']['f1'] for r in rs], 0).round(3).tolist())
    print('   top confusions (true->pred, seed-averaged logits)', top_confusions(Yva_np, mean_lg.argmax(1)))


def boot(y, pairs, metric, desc):
    rng = np.random.default_rng(0)
    d = np.empty(CFG['bootstrap'])
    for i in tqdm(range(len(d)), desc=desc, leave=False):
        j = rng.integers(0, len(y), len(y))
        d[i] = np.mean([metric(y[j], q[j]) - metric(y[j], p[j]) for p, q in pairs])
    return np.percentile(d, [2.5, 97.5])


def paired_test(y, pa, pb, metric, desc):
    per_seed = [metric(y, q) - metric(y, p) for p, q in zip(pa, pb)]
    lo, hi = boot(y, list(zip(pa, pb)), metric, desc)
    gain = float(np.mean(per_seed))
    passed = gain >= CFG['adopt_min_gain'] and all(x > 0 for x in per_seed) and lo > 0
    return gain, per_seed, lo, hi, passed


def ens_test(y, ea, eb, metric, desc):
    gain = metric(y, eb) - metric(y, ea)
    lo, hi = boot(y, [(ea, eb)], metric, desc)
    return gain, lo, hi, gain >= CFG['adopt_min_gain'] and lo > 0


def compare(a, b, la_a=False, la_b=False):
    sa, pa, la_ = preds_of(a, la_a)
    sb, pb, lb_ = preds_of(b, la_b)
    common = sorted(set(sa) & set(sb))
    if not common:
        print(f'skip {a} vs {b}: no shared seeds')
        return
    ia, ib = [sa.index(s) for s in common], [sb.index(s) for s in common]
    pa, pb = [pa[i] for i in ia], [pb[i] for i in ib]
    ea = (np.mean([la_[i] for i in ia], 0) + (LA if la_a else 0)).argmax(1)
    eb = (np.mean([lb_[i] for i in ib], 0) + (LA if la_b else 0)).argmax(1)
    y = Yva_np
    tag = f"{b}{'+LA' if la_b else ''} vs {a}{'+LA' if la_a else ''}"
    g, ps, lo, hi, f1_pass = paired_test(y, pa, pb, mf1, f'F1 {tag}')
    ga, psa, loa, hia, acc_pass = paired_test(y, pa, pb, acc_of, f'acc {tag}')
    ge, loe, hie, ens_f1_pass = ens_test(y, ea, eb, mf1, f'ens F1 {tag}')
    gea, loea, hiea, ens_acc_pass = ens_test(y, ea, eb, acc_of, f'ens acc {tag}')
    if f1_pass or ens_f1_pass:
        verdict = 'ADOPT (macro-F1)'
    elif ((g >= 0 and hi > 0) or (ge >= 0 and hie > 0)) and (acc_pass or ens_acc_pass):
        verdict = 'ADOPT (macro-F1 tie, accuracy)'
    else:
        size_a, size_b = runs_of(a)[0]['params'], runs_of(b)[0]['params']
        verdict = 'not adopted' + (' (tie -> smaller model)' if size_b > size_a else '')
    print(f"{tag} | seeds {common}\n"
          f"   per-seed macroF1 {100 * g:+.2f} pts {[round(100 * x, 2) for x in ps]} CI [{100 * lo:+.2f}, {100 * hi:+.2f}] | "
          f"acc {100 * ga:+.2f} {[round(100 * x, 2) for x in psa]} CI [{100 * loa:+.2f}, {100 * hia:+.2f}]\n"
          f"   ensemble macroF1 {100 * ge:+.2f} pts CI [{100 * loe:+.2f}, {100 * hie:+.2f}] | "
          f"acc {100 * gea:+.2f} CI [{100 * loea:+.2f}, {100 * hiea:+.2f}]\n"
          f"   -> {verdict}")


for a, b, la_a, la_b in CFG['comparisons']:
    if runs_of(a) and runs_of(b):
        compare(a, b, la_a, la_b)

r18_in           seeds [0, 1, 2] | macroF1 0.8689±0.0063 | acc 0.9196±0.0046 | LA macroF1 0.8694 | ens macroF1 0.8818 acc 0.9299 | params 11,182,155 | 2.6s/ep
   per-class F1 [0.98, 0.968, 0.936, 0.969, 0.811, 0.519, 0.84, 0.591, 0.972, 0.995, 0.976]
   top confusions (true->pred, seed-averaged logits) ['5->7:47', '7->5:32', '6->4:20', '4->6:14', '1->8:7', '3->10:7']
scratch          seeds [0, 1, 2] | macroF1 0.8639±0.0030 | acc 0.9135±0.0023 | LA macroF1 0.8675 | ens macroF1 0.8720 acc 0.9193 | params 308,203 | 0.8s/ep
   per-class F1 [0.974, 0.972, 0.91, 0.945, 0.839, 0.489, 0.855, 0.588, 0.981, 0.989, 0.962]
   top confusions (true->pred, seed-averaged logits) ['5->7:49', '7->5:33', '6->4:23', '2->10:19', '4->6:12', '3->10:11']
r18_in_lr003     seeds [0, 1, 2] | macroF1 0.8673±0.0035 | acc 0.9163±0.0003 | LA macroF1 0.8672 | ens macroF1 0.8769 acc 0.9248 | params 11,182,155 | 1.6s/ep
   per-class F1 [0.964, 0.961, 0.925, 0.974, 0.827, 0.539, 0.851, 0.571, 0.967, 0.988, 0.973]
   top

F1 r18_in_lr03_mixup vs r18_in_lr03:   0%|          | 0/2000 [00:00<?, ?it/s]

acc r18_in_lr03_mixup vs r18_in_lr03:   0%|          | 0/2000 [00:00<?, ?it/s]

ens F1 r18_in_lr03_mixup vs r18_in_lr03:   0%|          | 0/2000 [00:00<?, ?it/s]

ens acc r18_in_lr03_mixup vs r18_in_lr03:   0%|          | 0/2000 [00:00<?, ?it/s]

r18_in_lr03_mixup vs r18_in_lr03 | seeds [0, 1, 2]
   per-seed macroF1 -1.65 pts [-2.29, -0.85, -1.8] CI [-2.63, -0.64] | acc -1.46 [-1.7, -0.85, -1.83] CI [-2.08, -0.84]
   ensemble macroF1 -0.16 pts CI [-1.46, +1.14] | acc -0.72 CI [-1.49, +0.08]
   -> not adopted


... (6 outputs left out)

acc r50_in_lr03 vs r18_in_lr03:   0%|          | 0/2000 [00:00<?, ?it/s]

ens F1 r50_in_lr03 vs r18_in_lr03:   0%|          | 0/2000 [00:00<?, ?it/s]

ens acc r50_in_lr03 vs r18_in_lr03:   0%|          | 0/2000 [00:00<?, ?it/s]

r50_in_lr03 vs r18_in_lr03 | seeds [0, 1, 2]
   per-seed macroF1 -0.78 pts [-1.02, -1.1, -0.23] CI [-1.83, +0.25] | acc -0.82 [-1.1, -0.51, -0.85] CI [-1.40, -0.23]
   ensemble macroF1 +0.25 pts CI [-1.12, +1.63] | acc -0.13 CI [-0.89, +0.68]
   -> not adopted (tie -> smaller model)


In [50]:
# Cell 9: Final fit and export of ensemble members
assert CFG['final_model'] is not None, 'set CFG["final_model"] (and final_la) in Cell 2 after reading Cell 8, rerun Cell 2'
name = CFG['final_model']
c = CFG['models'][name]
if CFG['final_fit'] == 'train+valid':
    Xf, Yf = torch.cat([Xtr, Xva]), torch.cat([Ytr, Yva])
else:
    Xf, Yf = Xtr, Ytr
prior_f = np.bincount(Yf.cpu().numpy(), minlength=11) / len(Yf)
old = glob.glob(os.path.join(CFG['sub_dir'], 'member_*.pt'))
assert len(old) <= len(CFG['final_seeds']), f'stale member files in {CFG["sub_dir"]}: {old}'
for k, seed in enumerate(tqdm(CFG['final_seeds'], desc='final members')):
    net, info = train_model(name, seed, Xf, Yf)
    if CFG['final_la']:
        net.logit_bias.copy_(torch.tensor(-CFG['la_tau'] * np.log(prior_f), dtype=torch.float32, device=dev))
    path = os.path.join(CFG['sub_dir'], f'member_{k}.pt')
    torch.save(dict(arch=c['arch'], size=c['size'], dropout=c['dropout'], num_classes=CFG['num_classes'],
                    state_dict={kk: v.detach().cpu() for kk, v in net.state_dict().items()},
                    train_info=dict(model=name, seed=seed, final_fit=CFG['final_fit'], final_la=CFG['final_la'],
                                    n_train=len(Yf), train_loss=info['train_loss'], sec=info['sec'])), path)
    tqdm.write(f"SAVED {path} | seed {seed} | {info['sec']:.0f}s | params {info['params']:,} | "
               f"train loss {info['train_loss']:.4f} | {os.path.getsize(path) / 2**20:.1f} MB")
    del net
    torch.cuda.empty_cache()
print('members:', sorted(os.listdir(CFG['sub_dir'])))

AssertionError: set CFG["final_model"] (and final_la) in Cell 2 after reading Cell 8, rerun Cell 2

In [ ]:
# Cell 10: Submission check: predict.py on valid (GPU and CPU)
assert os.path.isfile(os.path.join(CFG['sub_dir'], 'predict.py')), 'upload predict.py into ' + CFG['sub_dir']
for d in tqdm(['cuda', 'cpu'], desc='predict check'):
    os.environ['PREDICT_DEVICE'] = d
    spec = importlib.util.spec_from_file_location(f'predict_{d}', os.path.join(CFG['sub_dir'], 'predict.py'))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    t0 = time.time()
    p = mod.predict(Xva_np.copy())
    dt = time.time() - t0
    assert isinstance(p, np.ndarray) and p.shape == (len(Xva_np),) and p.dtype.kind == 'i' and p.min() >= 0 and p.max() <= 10
    s = scores(Yva_np, p)
    tqdm.write(f"{d}: {dt:.1f}s for {len(p)} images | macroF1 {s['macro_f1']:.4f} acc {s['acc']:.4f} | "
               f"params {sum(sum(q.numel() for q in n.parameters()) for n in mod._load()[1]):,}")
os.environ.pop('PREDICT_DEVICE')
if CFG['final_fit'] == 'train+valid':
    print('NOTE: final members were trained on valid; this valid score is a wiring check, not an estimate')

In [ ]:
# Cell 11: Mixed ensemble check (r18_in_lr03 x3 + scratch x3, probability averaging) from saved valid logits
def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


base_l = preds_of('r18_in_lr03', False)[2]
scr_l = preds_of('scratch', False)[2]
assert len(base_l) == 3 and len(scr_l) == 3, 'need seeds 0-2 for r18_in_lr03 and scratch'
y = Yva_np
e_base = np.mean(base_l, 0).argmax(1)
e_base_p = np.mean([softmax(l) for l in base_l], 0).argmax(1)
e_mix = np.mean([softmax(l) for l in base_l + scr_l], 0).argmax(1)
for tag, e in [('r18_lr03 x3, logit avg (current)', e_base),
               ('r18_lr03 x3, prob avg', e_base_p),
               ('r18_lr03 x3 + scratch x3, prob avg', e_mix)]:
    s = scores(y, e)
    print(f"{tag:36s} macroF1 {s['macro_f1']:.4f} acc {s['acc']:.4f} | per-class {np.round(s['f1'], 3).tolist()}")
    print(f"{'':36s} top confusions {top_confusions(y, e)}")
ge, loe, hie, f1_pass = ens_test(y, e_base, e_mix, mf1, 'ens F1 mix vs current')
gea, loea, hiea, acc_pass = ens_test(y, e_base, e_mix, acc_of, 'ens acc mix vs current')
if f1_pass:
    verdict = 'ADOPT (macro-F1)'
elif ge >= 0 and hie > 0 and acc_pass:
    verdict = 'ADOPT (macro-F1 tie, accuracy)'
else:
    verdict = 'not adopted'
print(f"mix vs current | macroF1 {100 * ge:+.2f} pts CI [{100 * loe:+.2f}, {100 * hie:+.2f}] | "
      f"acc {100 * gea:+.2f} CI [{100 * loea:+.2f}, {100 * hiea:+.2f}] -> {verdict}")
print(f"params: current {3 * 11182155:,} | mix {3 * 11182155 + 3 * 308203:,}")

In [ ]:
# Cell 12: Test-time shift averaging check (r18_in_lr03, train only, 3 seeds, same models plain vs 5-shift)
TTA_SHIFTS = [(0, 0), (1, 0), (-1, 0), (0, 1), (0, -1)]


def shift_batch(x, dx, dy):
    if dx == 0 and dy == 0:
        return x
    theta = torch.tensor([[1.0, 0.0, -2.0 * dx / 28], [0.0, 1.0, -2.0 * dy / 28]], dtype=x.dtype, device=x.device)
    theta = theta.unsqueeze(0).expand(x.shape[0], 2, 3)
    return F.grid_sample(x, F.affine_grid(theta, list(x.shape), align_corners=False),
                         mode='bilinear', padding_mode='border', align_corners=False)


@torch.no_grad()
def tta_logits(net, X, shifts, bs=1024):
    net.eval()
    out = []
    for i in range(0, len(X), bs):
        xb = X[i:i + bs]
        out.append(torch.stack([net(shift_batch(xb, dx, dy)).float() for dx, dy in shifts]).mean(0))
    return torch.cat(out).cpu().numpy()


plain, tta = [], []
for seed in tqdm(CFG['seeds'], desc='tta models'):
    net, info = train_model('r18_in_lr03', seed, Xtr, Ytr)
    plain.append(get_logits(net, Xva))
    tta.append(tta_logits(net, Xva, TTA_SHIFTS))
    del net
    torch.cuda.empty_cache()
np.savez(os.path.join(CFG['out_dir'], 'tta_logits.npz'), plain=np.stack(plain), tta=np.stack(tta))
y = Yva_np
for tag, ls in [('plain', plain), ('tta5', tta)]:
    ps = [scores(y, l.argmax(1)) for l in ls]
    se = scores(y, np.mean(ls, 0).argmax(1))
    print(f"{tag:6s} per-seed macroF1 {np.mean([p['macro_f1'] for p in ps]):.4f} acc {np.mean([p['acc'] for p in ps]):.4f} | "
          f"ens macroF1 {se['macro_f1']:.4f} acc {se['acc']:.4f} | ens per-class {np.round(se['f1'], 3).tolist()}")
pa, pb = [l.argmax(1) for l in plain], [l.argmax(1) for l in tta]
g, ps_, lo, hi, f1_pass = paired_test(y, pa, pb, mf1, 'tta F1')
ga, psa, loa, hia, acc_pass = paired_test(y, pa, pb, acc_of, 'tta acc')
ea, eb = np.mean(plain, 0).argmax(1), np.mean(tta, 0).argmax(1)
ge, loe, hie, ens_f1_pass = ens_test(y, ea, eb, mf1, 'tta ens F1')
gea, loea, hiea, ens_acc_pass = ens_test(y, ea, eb, acc_of, 'tta ens acc')
if f1_pass or ens_f1_pass:
    verdict = 'ADOPT (macro-F1)'
elif ((g >= 0 and hi > 0) or (ge >= 0 and hie > 0)) and (acc_pass or ens_acc_pass):
    verdict = 'ADOPT (macro-F1 tie, accuracy)'
else:
    verdict = 'not adopted'
print(f"tta5 vs plain | per-seed macroF1 {100 * g:+.2f} {[round(100 * x, 2) for x in ps_]} CI [{100 * lo:+.2f}, {100 * hi:+.2f}] | "
      f"acc {100 * ga:+.2f} CI [{100 * loa:+.2f}, {100 * hia:+.2f}]\n"
      f"              ensemble macroF1 {100 * ge:+.2f} CI [{100 * loe:+.2f}, {100 * hie:+.2f}] | acc {100 * gea:+.2f} "
      f"CI [{100 * loea:+.2f}, {100 * hiea:+.2f}] -> {verdict}")